# Day 2: Evaluation, Feedback & Production Readiness

## Advanced Multi-Agent AI Systems with LangChain & LangGraph
### For Support & Meta Engineers

---

## 🎯 What You'll Learn

In this advanced lab, you will:
1. Build a **5-agent Root Cause Analysis (RCA) system**
2. Implement **advanced guardrails** (loop detection, tool allowlists, time budgets)
3. Create an **evaluation framework** with precision/recall/F1 metrics
4. Handle **failure modes** and implement recovery patterns
5. Apply **production deployment** best practices
6. Perform **meta-engineering** analysis on system behavior

## 📊 What You'll Build

A **5-agent RCA system** that:
- **Parses logs** and extracts structured data
- **Detects patterns** and anomalies
- **Correlates events** across time windows
- **Generates hypotheses** with evidence
- **Validates** hypotheses against contradicting evidence

All with **production-grade guardrails** and **comprehensive evaluation**.

---

## Part 1: Setup & Recap

### 1.1 Import Dependencies

In [ ]:
# Install project dependencies (safe to re-run)
# Optimized for Day 2 - removed unnecessary packages
%pip install -q \
    langchain>=0.1.0 \
    langchain-openai>=0.0.5 \
    langgraph>=0.0.20 \
    langchain-core>=0.1.0 \
    openai>=1.12.0 \
    python-dotenv>=1.0.0 \
    pydantic>=2.0.0 \
    pandas>=2.0.0 \
    numpy>=1.24.0 \
    scikit-learn>=1.3.0 \
    typing-extensions>=4.5.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"✓ Project root: {project_root}")
print(f"✓ Day 2: Evaluation & Production Readiness")

✓ Project root: c:\Users\bharg\Downloads\Work\advanced-multi-agent-ai-systems-2-half-days
✓ Day 2: Evaluation & Production Readiness


### 1.2 Initialize LLM & Load Data

In [3]:
from src.llm_factory import get_llm, print_llm_stats

llm = get_llm(force_mock=False, deterministic=True, verbose=True)

incidents = []
with open(project_root / 'data' / 'incidents_large_small.jsonl', 'r') as f:
    for line in f:
        incidents.append(json.loads(line.strip()))

with open(project_root / 'data' / 'sample_logs.txt', 'r') as f:
    sample_logs = f.read()

metrics_df = pd.read_csv(project_root / 'data' / 'metrics_samples.csv')

print(f"\n✓ Loaded {len(incidents)} incidents")
print(f"✓ Loaded {len(sample_logs.splitlines())} log lines")
print(f"✓ Loaded {len(metrics_df)} metric data points")

🚀 Running with ChatOpenAI
   Model: gpt-4o-mini

✓ Loaded 20 incidents
✓ Loaded 49 log lines
✓ Loaded 90 metric data points


---

## Part 2: Root Cause Analysis System

### 2.1 RCA Workflow Architecture

```
┌─────────────┐
│ Parse Logs  │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│   Detect    │
│  Patterns   │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│  Correlate  │
│   Events    │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│ Guardrails  │ ← Max steps, loop detection
└──────┬──────┘
       │
       ▼
┌─────────────┐
│  Generate   │
│ Hypothesis  │
└──────┬──────┘
       │
       ▼
┌─────────────┐
│  Validate   │
│ Hypothesis  │
└─────────────┘
```

### 2.2 Initialize RCA Workflow

In [4]:
from src.observability import TraceLogger
from src.workflow_day2 import create_rca_workflow

logger = TraceLogger(log_file="traces_day2.jsonl")
logger.clear()

rca_workflow = create_rca_workflow(
    llm=llm,
    logger=logger,
    max_steps=15,
    log_data=sample_logs
)

print("✓ RCA workflow created")
print(f"  Max steps: 15")
print(f"  Log data: {len(sample_logs)} characters")

✓ RCA workflow created
  Max steps: 15
  Log data: 6121 characters


### 2.3 Run RCA on Single Incident

In [5]:
test_incident = incidents[0]

print("\n" + "="*60)
print(f"Running RCA for: {test_incident['id']}")
print(f"Title: {test_incident['title']}")
print("="*60 + "\n")

rca_result = rca_workflow.run(test_incident, sample_logs)

print("\n" + "="*60)
print("RCA Result:")
print("="*60)
print(f"Status: {rca_result['status']}")
print(f"Trace ID: {rca_result['trace_id']}")
print(f"Steps: {rca_result['step_count']}")
print(f"Errors: {len(rca_result.get('errors', []))}")


Running RCA for: INC-10001
Title: Database Connection Pool Exhausted


RCA Result:
Status: completed_with_errors
Trace ID: 54999c8f-3737-44d1-abea-4c708557700c
Steps: 2
Errors: 1


### 2.4 Examine Log Analysis

In [6]:
log_analysis = rca_result.get('log_analysis', {})

print("\n📊 LOG ANALYSIS")
print("="*60)
if log_analysis:
    print(json.dumps(log_analysis, indent=2))
else:
    print("No log analysis available")


📊 LOG ANALYSIS
{
  "total_lines": 49,
  "error_count": 32,
  "warning_count": 10,
  "error_types": {
    "connection_timeout": 16,
    "network_error": 2,
    "null_pointer": 2,
    "resource_exhausted": 1
  },
  "affected_services": [
    "service_monitoring",
    "service_cache",
    "service_storage",
    "service_payment",
    "service_report",
    "service_api",
    "service_auth",
    "service_ops",
    "service_gateway",
    "service_notification"
  ],
  "time_range": {
    "start": "2024-01-27T10:15:00",
    "end": "2024-01-27T10:19:00"
  },
  "sample_errors": [
    "2024-01-27T10:15:00 ERROR service_api [RequestHandler] Database connection failed: java.sql.SQLException: Cannot get a connection, pool error Timeout waiting for idle object",
    "2024-01-27T10:15:05 ERROR service_api [RequestHandler] Database connection failed: java.sql.SQLException: Cannot get a connection, pool error Timeout waiting for idle object",
    "2024-01-27T10:15:10 CRITICAL service_api [HealthCheck] 

### 2.5 Examine Pattern Detection

In [7]:
patterns = rca_result.get('patterns', [])

print("\n📊 PATTERNS DETECTED")
print("="*60)
if patterns:
    for i, pattern in enumerate(patterns, 1):
        print(f"\nPattern {i}:")
        print(json.dumps(pattern, indent=2))
else:
    print("No patterns detected")


📊 PATTERNS DETECTED
No patterns detected


### 2.6 Examine Hypothesis & Validation

In [8]:
hypothesis = rca_result.get('hypothesis', {})
validation = rca_result.get('validation', {})

print("\n📊 ROOT CAUSE HYPOTHESIS")
print("="*60)
if hypothesis:
    print(f"Hypothesis: {hypothesis.get('hypothesis', 'N/A')}")
    print(f"Confidence: {hypothesis.get('confidence', 0):.2%}")
    print(f"\nRecommended Actions:")
    for i, action in enumerate(hypothesis.get('recommended_actions', []), 1):
        print(f"  {i}. {action}")
    print(f"\nEstimated Resolution: {hypothesis.get('estimated_resolution_time_hours', 'N/A')} hours")
else:
    print("No hypothesis generated")

print("\n📊 VALIDATION RESULT")
print("="*60)
if validation:
    print(f"Valid: {validation.get('is_valid', False)}")
    print(f"Confidence: {validation.get('confidence', 0):.2%}")
    print(f"Evidence Strength: {validation.get('evidence_strength', 0):.2%}")
    print(f"Risk Level: {validation.get('risk_level', 'unknown')}")
    print(f"Recommendation: {validation.get('recommendation', 'N/A')}")
else:
    print("No validation result available")


📊 ROOT CAUSE HYPOTHESIS
No hypothesis generated

📊 VALIDATION RESULT
No validation result available


---

## Part 3: Advanced Guardrails

### 3.1 Loop Detection

**Problem**: Agent gets stuck repeating the same node

**Solution**: Track node execution counts and terminate if threshold exceeded

In [9]:
print("\n" + "="*60)
print("Testing Loop Detection Guardrail")
print("="*60 + "\n")

print("Loop detection is built into the workflow.")
print("Each node tracks execution count in state['loop_detection']")
print("\nIf any node executes > 3 times, a warning is logged.")
print("This prevents infinite loops in evidence gathering.")

trace_summary = logger.get_trace_summary(rca_result['trace_id'])
print(f"\nCurrent trace violations: {trace_summary.get('violations_count', 0)}")


Testing Loop Detection Guardrail

Loop detection is built into the workflow.
Each node tracks execution count in state['loop_detection']

If any node executes > 3 times, a warning is logged.
This prevents infinite loops in evidence gathering.

Current trace violations: 0


### 3.2 Tool Allowlist Guardrail

**Problem**: Agent attempts to use unauthorized tools

**Solution**: Whitelist only approved tools per agent

In [10]:
from src.tools import get_read_only_tools, create_tool_allowlist

all_tools = get_read_only_tools(
    incident_db=incidents,
    log_data=sample_logs
)

print("\n📦 ALL AVAILABLE TOOLS")
print("="*60)
for tool in all_tools:
    print(f"  - {tool.name}")

allowed_tools = create_tool_allowlist(
    all_tools,
    allowed_tool_names=['search_incidents', 'query_logs']
)

print("\n📦 ALLOWED TOOLS (Allowlist)")
print("="*60)
for tool in allowed_tools:
    print(f"  - {tool.name}")

print("\n✓ Tool allowlist prevents unauthorized tool usage")
print("  This is critical for production safety!")


📦 ALL AVAILABLE TOOLS
  - search_incidents
  - lookup_policy
  - query_logs
  - query_metrics

📦 ALLOWED TOOLS (Allowlist)
  - search_incidents
  - query_logs

✓ Tool allowlist prevents unauthorized tool usage
  This is critical for production safety!


### 3.3 Time & Cost Budget Guardrails

In [11]:
from src.observability import PerformanceMonitor

perf_monitor = PerformanceMonitor()

print("\n⏱️  TIME & COST BUDGETS")
print("="*60)
print("\nProduction guardrails should include:")
print("  1. Max execution time (e.g., 30 seconds)")
print("  2. Max token budget (e.g., 10,000 tokens)")
print("  3. Max cost budget (e.g., $0.10 per incident)")
print("\nImplementation:")
print("  - Track start time in workflow state")
print("  - Check elapsed time in guardrails node")
print("  - Terminate if budget exceeded")

trace_summary = logger.get_trace_summary(rca_result['trace_id'])
duration_ms = trace_summary.get('total_duration_ms', 0)

print(f"\nCurrent RCA execution time: {duration_ms:.2f}ms")
print(f"Within 30s budget: {duration_ms < 30000}")


⏱️  TIME & COST BUDGETS

Production guardrails should include:
  1. Max execution time (e.g., 30 seconds)
  2. Max token budget (e.g., 10,000 tokens)
  3. Max cost budget (e.g., $0.10 per incident)

Implementation:
  - Track start time in workflow state
  - Check elapsed time in guardrails node
  - Terminate if budget exceeded

Current RCA execution time: 6228.78ms
Within 30s budget: True


---

## Part 4: Failure Modes & Recovery

### 4.1 Failure Mode 1: Invalid JSON Response

**Problem**: LLM returns malformed JSON

**Recovery**: Catch JSONDecodeError, log error, continue with degraded functionality

In [12]:
print("\n" + "="*60)
print("Failure Mode 1: Invalid JSON")
print("="*60 + "\n")

print("Built-in handling:")
print("  1. Try to parse JSON response")
print("  2. If JSONDecodeError, log error to state['errors']")
print("  3. Continue workflow with empty result for that node")
print("  4. Downstream nodes check for missing data")
print("\nThis prevents total workflow failure from single node error.")

if rca_result.get('errors'):
    print(f"\nErrors in current run: {len(rca_result['errors'])}")
    for i, error in enumerate(rca_result['errors'][:3], 1):
        print(f"  {i}. {error[:80]}...")
else:
    print("\n✓ No errors in current run")


Failure Mode 1: Invalid JSON

Built-in handling:
  1. Try to parse JSON response
  2. If JSONDecodeError, log error to state['errors']
  3. Continue workflow with empty result for that node
  4. Downstream nodes check for missing data

This prevents total workflow failure from single node error.

Errors in current run: 1
  1. Invalid JSON from pattern detection: Expecting value: line 1 column 1 (char 0)...


### 4.2 Failure Mode 2: Runaway Evidence Gathering

**Problem**: Agent loops infinitely collecting evidence

**Recovery**: Loop detection + max steps guardrail

In [13]:
print("\n" + "="*60)
print("Failure Mode 2: Runaway Evidence Gathering")
print("="*60 + "\n")

print("Protection mechanisms:")
print("  1. Max steps guardrail (e.g., 15 steps)")
print("  2. Loop detection (track node execution counts)")
print("  3. Conditional edges (only proceed if conditions met)")
print("\nSimulating runaway scenario...")

limited_rca = create_rca_workflow(
    llm=llm,
    logger=logger,
    max_steps=5,
    log_data=sample_logs
)

result_limited = limited_rca.run(incidents[1], sample_logs)

print(f"\nResult with max_steps=5:")
print(f"  Status: {result_limited['status']}")
print(f"  Steps executed: {result_limited['step_count']}")
print(f"  Terminated early: {result_limited['step_count'] >= 5}")


Failure Mode 2: Runaway Evidence Gathering

Protection mechanisms:
  1. Max steps guardrail (e.g., 15 steps)
  2. Loop detection (track node execution counts)
  3. Conditional edges (only proceed if conditions met)

Simulating runaway scenario...

Result with max_steps=5:
  Status: completed_with_errors
  Steps executed: 2
  Terminated early: False


### 4.3 Failure Mode 3: Tool Misuse

**Problem**: Agent attempts write operation without approval

**Recovery**: Tool allowlist + approval workflow

In [15]:
from src.tools import EscalateIncidentTool

print("\n" + "="*60)
print("Failure Mode 3: Unauthorized Tool Usage")
print("="*60 + "\n")

escalate_tool = EscalateIncidentTool(require_approval=True)

print("Testing escalation tool (requires approval):")
result = escalate_tool.invoke({
    "incident_id": "INC-10001",
    "reason": "Critical P0 incident",
    "target_team": "Engineering Manager"
})

result_data = json.loads(result)
print(f"\nResult: {result_data['status']}")
print(f"Approval required: {result_data.get('approval_required', False)}")
print(f"Message: {result_data.get('message', 'N/A')}")

print("\n✓ Write operations blocked until human approval")
print("  This prevents automated actions without oversight")


Failure Mode 3: Unauthorized Tool Usage

Testing escalation tool (requires approval):

Result: pending_approval
Approval required: True
Message: Escalation requires human approval. Request submitted.

✓ Write operations blocked until human approval
  This prevents automated actions without oversight


---

## Part 5: Evaluation Framework

### 5.1 Create Evaluation Cases

In [16]:
from src.evaluation import create_evaluation_cases, Evaluator

eval_cases = create_evaluation_cases()

print("\n📊 EVALUATION TEST CASES")
print("="*60)
print(f"Total cases: {len(eval_cases)}\n")

for i, case in enumerate(eval_cases, 1):
    print(f"{i}. {case.incident_id}")
    print(f"   Ground Truth: {case.ground_truth_severity.value} - {case.ground_truth_category}")
    print(f"   Team: {case.ground_truth_team}")
    print()


📊 EVALUATION TEST CASES
Total cases: 5

1. EVAL-001
   Ground Truth: P0 - Database
   Team: Database Team

2. EVAL-002
   Ground Truth: P1 - Network
   Team: Network Operations

3. EVAL-003
   Ground Truth: P2 - Application
   Team: Application Support

4. EVAL-004
   Ground Truth: P3 - Infrastructure
   Team: Infrastructure

5. EVAL-005
   Ground Truth: P1 - Security
   Team: Security Team



### 5.2 Run Evaluation

In [17]:
from src.workflow_day1 import create_triage_workflow

evaluator = Evaluator()

eval_workflow = create_triage_workflow(
    llm=llm,
    logger=logger,
    max_steps=10,
    incident_db=incidents
)

print("\n" + "="*60)
print("Running Evaluation")
print("="*60 + "\n")

for i, case in enumerate(eval_cases, 1):
    print(f"{i}. Evaluating {case.incident_id}...")
    
    incident_data = {
        'id': case.incident_id,
        'title': case.description[:50],
        'description': case.description
    }
    
    start_time = datetime.now()
    result = eval_workflow.run(incident_data)
    latency_ms = (datetime.now() - start_time).total_seconds() * 1000
    
    classification = result.get('classification', {})
    routing = result.get('routing', {})
    
    evaluator.add_result(
        incident_id=case.incident_id,
        predicted_severity=classification.get('severity'),
        predicted_category=classification.get('category'),
        predicted_team=routing.get('assigned_team'),
        ground_truth=case,
        latency_ms=latency_ms,
        success=(result['status'] == 'completed')
    )
    
    print(f"   ✓ Completed in {latency_ms:.0f}ms\n")

print("✓ Evaluation complete")


Running Evaluation

1. Evaluating EVAL-001...
   ✓ Completed in 4008ms

2. Evaluating EVAL-002...
   ✓ Completed in 5526ms

3. Evaluating EVAL-003...
   ✓ Completed in 5662ms

4. Evaluating EVAL-004...
   ✓ Completed in 4139ms

5. Evaluating EVAL-005...
   ✓ Completed in 3929ms

✓ Evaluation complete


### 5.3 View Evaluation Results

In [18]:
results_df = evaluator.get_results_dataframe()

print("\n📊 EVALUATION RESULTS")
print("="*60)
display(results_df[[
    'incident_id',
    'predicted_severity',
    'ground_truth_severity',
    'predicted_category',
    'ground_truth_category',
    'latency_ms',
    'success'
]])


📊 EVALUATION RESULTS


,incident_id,predicted_severity,ground_truth_severity,predicted_category,ground_truth_category,latency_ms,success
0,EVAL-001,SeverityLevel.P1,P0,IncidentCategory.DATABASE,Database,4007.540,True
1,EVAL-002,SeverityLevel.P1,P1,IncidentCategory.INFRASTRUCTURE,Network,5525.721,True
2,EVAL-003,SeverityLevel.P2,P2,IncidentCategory.APPLICATION,Application,5661.765,False
3,EVAL-004,SeverityLevel.P3,P3,IncidentCategory.INFRASTRUCTURE,Infrastructure,4139.077,True
4,EVAL-005,SeverityLevel.P1,P1,IncidentCategory.SECURITY,Security,3928.976,False


### 5.4 Calculate Metrics

In [19]:
classification_eval = evaluator.evaluate_classification()
routing_eval = evaluator.evaluate_routing()
performance_eval = evaluator.evaluate_performance()

print("\n📊 CLASSIFICATION METRICS")
print("="*60)
print(f"Total Cases: {classification_eval.total_cases}")
print(f"Severity Accuracy: {classification_eval.accuracy_severity:.2%}")
print(f"Category Accuracy: {classification_eval.accuracy_category:.2%}")

print("\n📊 ROUTING METRICS")
print("="*60)
print(f"Total Cases: {routing_eval.total_cases}")
print(f"Team Assignment Accuracy: {routing_eval.accuracy:.2%}")

print("\n📊 PERFORMANCE METRICS")
print("="*60)
print(f"Average Latency: {performance_eval.avg_latency_ms:.2f}ms")
print(f"P50 Latency: {performance_eval.p50_latency_ms:.2f}ms")
print(f"P95 Latency: {performance_eval.p95_latency_ms:.2f}ms")
print(f"P99 Latency: {performance_eval.p99_latency_ms:.2f}ms")
print(f"Success Rate: {performance_eval.success_rate:.2%}")


📊 CLASSIFICATION METRICS
Total Cases: 3
Severity Accuracy: 66.67%
Category Accuracy: 66.67%

📊 ROUTING METRICS
Total Cases: 3
Team Assignment Accuracy: 33.33%

📊 PERFORMANCE METRICS
Average Latency: 4652.62ms
P50 Latency: 4139.08ms
P95 Latency: 5661.77ms
P99 Latency: 5661.77ms
Success Rate: 60.00%


### 5.5 Generate Evaluation Report

In [20]:
report = evaluator.generate_report()
print(report)

EVALUATION REPORT

CLASSIFICATION METRICS
----------------------------------------------------------------------
Total Cases: 3
Severity Accuracy: 66.67%
Category Accuracy: 66.67%

ROUTING METRICS
----------------------------------------------------------------------
Total Cases: 3
Team Assignment Accuracy: 33.33%

PERFORMANCE METRICS
----------------------------------------------------------------------
Average Latency: 4652.62ms
P50 Latency: 4139.08ms
P95 Latency: 5661.77ms
P99 Latency: 5661.77ms
Success Rate: 60.00%



---

## Part 6: Production Deployment Patterns

### 6.1 Deployment Checklist

In [21]:
deployment_checklist = {
    'Security': [
        '☐ API keys in environment variables only',
        '☐ No secrets in code or logs',
        '☐ Tool allowlists configured',
        '☐ Write operations require approval',
        '☐ Audit logging enabled'
    ],
    'Guardrails': [
        '☐ Max steps configured',
        '☐ Loop detection enabled',
        '☐ Time budget set',
        '☐ Cost budget set',
        '☐ Error thresholds configured'
    ],
    'Observability': [
        '☐ JSONL trace logging',
        '☐ Trace IDs in all logs',
        '☐ Performance metrics tracked',
        '☐ Guardrail violations logged',
        '☐ Alerting configured'
    ],
    'Testing': [
        '☐ Unit tests for all nodes',
        '☐ Integration tests for workflows',
        '☐ Evaluation framework in place',
        '☐ Failure mode testing',
        '☐ Load testing completed'
    ],
    'Validation': [
        '☐ Pydantic schemas for all outputs',
        '☐ Schema validation enabled',
        '☐ Confidence thresholds set',
        '☐ Evidence validation',
        '☐ Human review for low confidence'
    ]
}

print("\n📋 PRODUCTION DEPLOYMENT CHECKLIST")
print("="*60)
for category, items in deployment_checklist.items():
    print(f"\n{category}:")
    for item in items:
        print(f"  {item}")


📋 PRODUCTION DEPLOYMENT CHECKLIST

Security:
  ☐ API keys in environment variables only
  ☐ No secrets in code or logs
  ☐ Tool allowlists configured
  ☐ Write operations require approval
  ☐ Audit logging enabled

Guardrails:
  ☐ Max steps configured
  ☐ Loop detection enabled
  ☐ Time budget set
  ☐ Cost budget set
  ☐ Error thresholds configured

Observability:
  ☐ JSONL trace logging
  ☐ Trace IDs in all logs
  ☐ Performance metrics tracked
  ☐ Guardrail violations logged
  ☐ Alerting configured

Testing:
  ☐ Unit tests for all nodes
  ☐ Integration tests for workflows
  ☐ Evaluation framework in place
  ☐ Failure mode testing
  ☐ Load testing completed

Validation:
  ☐ Pydantic schemas for all outputs
  ☐ Schema validation enabled
  ☐ Confidence thresholds set
  ☐ Evidence validation
  ☐ Human review for low confidence


### 6.2 Monitoring Dashboard Data

In [22]:
all_traces = logger.get_all_traces_summary()

if not all_traces.empty:
    print("\n📊 MONITORING DASHBOARD")
    print("="*60)
    
    print(f"\nTotal Workflows Executed: {len(all_traces)}")
    print(f"Completed: {(all_traces['status'] == 'completed').sum()}")
    print(f"Failed: {(all_traces['status'] == 'failed').sum()}")
    
    print(f"\nPerformance:")
    print(f"  Avg Duration: {all_traces['total_duration_ms'].mean():.2f}ms")
    print(f"  P95 Duration: {all_traces['total_duration_ms'].quantile(0.95):.2f}ms")
    
    print(f"\nQuality:")
    print(f"  Total Errors: {all_traces['errors_count'].sum()}")
    print(f"  Total Violations: {all_traces['violations_count'].sum()}")
    print(f"  Total Tool Calls: {all_traces['tool_calls_count'].sum()}")
    
    print("\n📈 Trace Summary Table:")
    display(all_traces[[
        'workflow_name',
        'status',
        'total_duration_ms',
        'tool_calls_count',
        'errors_count',
        'violations_count'
    ]].head(10))
else:
    print("No trace data available")


📊 MONITORING DASHBOARD

Total Workflows Executed: 7
Completed: 7
Failed: 0

Performance:
  Avg Duration: 5523.30ms
  P95 Duration: 8290.85ms

Quality:
  Total Errors: 5
  Total Violations: 0
  Total Tool Calls: 0

📈 Trace Summary Table:


,workflow_name,status,total_duration_ms,tool_calls_count,errors_count,violations_count
0,RCAWorkflow,completed,9174.595118,0,2,0
1,RCAWorkflow,completed,6228.779078,0,1,0
2,TriageWorkflow,completed,3928.287029,0,1,0
3,TriageWorkflow,completed,4138.006926,0,0,0
4,TriageWorkflow,completed,4006.914854,0,0,0
5,TriageWorkflow,completed,5525.240183,0,0,0
6,TriageWorkflow,completed,5661.299944,0,1,0


---

## Part 7: Meta-Engineering Analysis

### 7.1 System Behavior Analysis

**Meta-engineering** = Analyzing the AI system's behavior to improve it

In [23]:
print("\n📊 META-ENGINEERING INSIGHTS")
print("="*60)

if not results_df.empty:
    print("\n1. Error Pattern Analysis:")
    error_rate_by_severity = results_df.groupby('ground_truth_severity')['success'].mean()
    print("   Success rate by severity:")
    for severity, rate in error_rate_by_severity.items():
        print(f"     {severity}: {rate:.1%}")
    
    print("\n2. Latency Analysis:")
    latency_by_category = results_df.groupby('ground_truth_category')['latency_ms'].mean()
    print("   Avg latency by category:")
    for category, latency in latency_by_category.items():
        print(f"     {category}: {latency:.0f}ms")
    
    print("\n3. Misclassification Analysis:")
    misclassified = results_df[
        results_df['predicted_severity'] != results_df['ground_truth_severity']
    ]
    if not misclassified.empty:
        print(f"   Total misclassifications: {len(misclassified)}")
        print("   Common patterns:")
        for _, row in misclassified.iterrows():
            print(f"     {row['incident_id']}: {row['ground_truth_severity']} → {row['predicted_severity']}")
    else:
        print("   ✓ No misclassifications!")
else:
    print("No evaluation data available")


📊 META-ENGINEERING INSIGHTS

1. Error Pattern Analysis:
   Success rate by severity:
     P0: 100.0%
     P1: 50.0%
     P2: 0.0%
     P3: 100.0%

2. Latency Analysis:
   Avg latency by category:
     Application: 5662ms
     Database: 4008ms
     Infrastructure: 4139ms
     Network: 5526ms
     Security: 3929ms

3. Misclassification Analysis:
   Total misclassifications: 1
   Common patterns:
     EVAL-001: P0 → SeverityLevel.P1


### 7.2 Recommendations for Improvement

In [24]:
print("\n💡 IMPROVEMENT RECOMMENDATIONS")
print("="*60)

recommendations = []

if classification_eval.accuracy_severity < 0.90:
    recommendations.append(
        "1. Improve severity classification:\n"
        "   - Add more examples to prompts\n"
        "   - Fine-tune on historical incidents\n"
        "   - Add confidence thresholds"
    )

if performance_eval.p95_latency_ms > 5000:
    recommendations.append(
        "2. Optimize latency:\n"
        "   - Cache common queries\n"
        "   - Parallelize independent nodes\n"
        "   - Use faster LLM for simple tasks"
    )

if not all_traces.empty and all_traces['errors_count'].sum() > 0:
    recommendations.append(
        "3. Reduce errors:\n"
        "   - Add retry logic with backoff\n"
        "   - Improve prompt engineering\n"
        "   - Add fallback responses"
    )

if recommendations:
    for rec in recommendations:
        print(f"\n{rec}")
else:
    print("\n✓ System performing well! No critical improvements needed.")
    print("\nOptional enhancements:")
    print("  - Add more evaluation test cases")
    print("  - Implement A/B testing framework")
    print("  - Add real-time monitoring dashboard")


💡 IMPROVEMENT RECOMMENDATIONS

1. Improve severity classification:
   - Add more examples to prompts
   - Fine-tune on historical incidents
   - Add confidence thresholds

2. Optimize latency:
   - Cache common queries
   - Parallelize independent nodes
   - Use faster LLM for simple tasks

3. Reduce errors:
   - Add retry logic with backoff
   - Improve prompt engineering
   - Add fallback responses


---

## Part 8: Final Statistics

### 8.1 LLM Usage

In [25]:
print_llm_stats(llm)


📊 LLM Usage Statistics
Model: ChatOpenAI
Note: Use LangSmith or callbacks for detailed tracking



### 8.2 Complete System Summary

In [27]:
print("\n" + "="*70)
print("COMPLETE SYSTEM SUMMARY")
print("="*70)

print("\n📊 WORKFLOWS EXECUTED")
print(f"  Day 1 Triage: {len([t for t in logger.traces if 'Triage' in t.get('workflow_name', '')])}")
print(f"  Day 2 RCA: {len([t for t in logger.traces if 'RCA' in t.get('workflow_name', '')])}")

print("\n📊 EVALUATION RESULTS")
print(f"  Classification Accuracy: {classification_eval.accuracy_severity:.1%}")
print(f"  Routing Accuracy: {routing_eval.accuracy:.1%}")
print(f"  Average Latency: {performance_eval.avg_latency_ms:.0f}ms")
print(f"  Success Rate: {performance_eval.success_rate:.1%}")

print("\n📊 GUARDRAILS")
if not all_traces.empty:
    print(f"  Total Violations: {all_traces['violations_count'].sum()}")
    print(f"  Total Errors: {all_traces['errors_count'].sum()}")
else:
    print("  No guardrail data available")

print("\n📊 PRODUCTION READINESS")
print("  ✓ Pydantic schemas validated")
print("  ✓ Guardrails implemented")
print("  ✓ Observability enabled")
print("  ✓ Evaluation framework complete")
print("  ✓ Failure modes tested")

print("\n" + "="*70)


COMPLETE SYSTEM SUMMARY

📊 WORKFLOWS EXECUTED
  Day 1 Triage: 5
  Day 2 RCA: 2

📊 EVALUATION RESULTS
  Classification Accuracy: 66.7%
  Routing Accuracy: 33.3%
  Average Latency: 4653ms
  Success Rate: 60.0%

📊 GUARDRAILS
  Total Violations: 0
  Total Errors: 5

📊 PRODUCTION READINESS
  ✓ Pydantic schemas validated
  ✓ Guardrails implemented
  ✓ Observability enabled
  ✓ Evaluation framework complete
  ✓ Failure modes tested



---

## 🎉 Course Complete!

Congratulations! You've built **production-ready multi-agent AI systems** with:

### ✅ Day 1 Achievements
- 3-agent incident triage system
- LangGraph StateGraph orchestration
- Pydantic structured outputs
- Basic guardrails
- JSONL trace logging

### ✅ Day 2 Achievements
- 5-agent root cause analysis system
- Advanced guardrails (loop detection, tool allowlists)
- Comprehensive evaluation framework
- Failure mode handling
- Production deployment patterns
- Meta-engineering analysis

### 🚀 Next Steps

1. **Deploy to production**:
   - Follow deployment checklist
   - Set up monitoring
   - Configure alerting

2. **Continuous improvement**:
   - Collect production data
   - Run regular evaluations
   - Iterate on prompts and workflows

3. **Scale up**:
   - Add more agents for specialized tasks
   - Implement parallel execution
   - Optimize for cost and latency

### 📚 Resources

- **LangChain Docs**: https://python.langchain.com/
- **LangGraph Docs**: https://langchain-ai.github.io/langgraph/
- **Pydantic Docs**: https://docs.pydantic.dev/

### 🙏 Thank You!

You're now equipped to build production-grade multi-agent AI systems for incident management, log analysis, and automated triage.

**Keep building amazing systems!** 🚀